In [2]:
import cv2
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub

model = hub.load("https://tfhub.dev/google/movenet/multipose/lightning/1")
movenet = model.signatures['serving_default']

2025-10-04 21:02:40.761354: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1759600963.414918   30964 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5797 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060, pci bus id: 0000:05:00.0, compute capability: 8.9


In [ ]:
# --- Pose detection by movenet ---
def detect_poses(frame):
    input_image = tf.image.resize_with_pad(tf.expand_dims(frame, axis=0), 256, 256)
    input_image = tf.cast(input_image, dtype=tf.int32)
    outputs = movenet(input_image)
    keypoints_with_scores = outputs['output_0'].numpy()  # shape: [1,6,56]
    return keypoints_with_scores[0]  # 6 people max


# --- Feature extraction per frame ---
def process_frame(frame, keypoints_with_scores, prev_people, fps, threshold=0.4):
    h, w, _ = frame.shape
    new_people = []
    features_out = []

    for person in keypoints_with_scores:
        scores = person[2::3]
        if np.sum(scores > threshold) < 5:
            continue

        # Reshape into (17, 3) = (y, x, conf)
        keypoints = np.array(person[:51]).reshape((17, 3))

        # Scale back to pixel coords
        keypoints[:, 0] *= h  # y
        keypoints[:, 1] *= w  # x

        new_people.append(keypoints)

        # Normalize coordinates to [-1, 1]
        y_norm = (keypoints[:, 0] / h) * 2 - 1
        x_norm = (keypoints[:, 1] / w) * 2 - 1
        conf = keypoints[:, 2]

        # Compute velocities
        person_velocities = np.zeros((17, 2))
        if prev_people:
            prev_keypoints = min(
                prev_people, key=lambda pk: np.linalg.norm(pk[:, :2] - keypoints[:, :2])
            )
            dt = 1.0 / fps
            for j, ((y, x, c), (py, px, pc)) in enumerate(zip(keypoints, prev_keypoints)):
                if c > threshold and pc > threshold:
                    vx, vy = (x - px) / dt, (y - py) / dt
                    # Normalize velocity
                    vx /= w
                    vy /= h
                    person_velocities[j] = [vx, vy]

        # Final feature vector per joint
        person_features = np.stack([x_norm, y_norm, conf,
                                    person_velocities[:, 0],
                                    person_velocities[:, 1]], axis=-1)  # (17, 5)

        features_out.append(person_features)

    return features_out, new_people


In [3]:

# --- Main video to numpy ---
def video_to_numpy(video_path, max_people=6, threshold=0.4):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    prev_people = []
    clip_features = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        keypoints_with_scores = detect_poses(rgb_frame)

        features_out, prev_people = process_frame(frame, keypoints_with_scores, prev_people, fps, threshold)

        # Ensure fixed number of people (pad if fewer, crop if more)
        while len(features_out) < max_people:
            features_out.append(np.zeros((17, 5)))  # pad missing person
        features_out = features_out[:max_people]

        clip_features.append(features_out)

    cap.release()

    # Shape: (T, P, V, F)
    clip_array = np.array(clip_features, dtype=np.float32)
    # T = num frames, P = max_people, V = 17 joints, F = 5 features
    return clip_array

In [8]:
import os
from tqdm import tqdm

In [5]:
def Save2Npy(file_dir, save_dir):
    """Transfer all the videos and save them into specified directory
    Args:
        file_dir: source folder of target videos
        save_dir: destination folder of output .npy files
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    # List the files
    videos = os.listdir(file_dir)
    for v in tqdm(videos):
        # Split video name
        video_name = v.split('.')[0]
        # Get src
        video_path = os.path.join(file_dir, v)
        # Get dest
        save_path = os.path.join(save_dir, video_name+'.npy')
        # Load and preprocess video
        data = video_to_numpy(video_path)
        # Save as .npy file
        np.save(save_path, data)

    return None

In [ ]:
source_path = 'data/'
target_path = 'transformed_data'

for f1 in ['non_violent', 'violent']:
    for f2 in ['cam1']:
        path1 = os.path.join(source_path, f1, f2)
        path2 = os.path.join(target_path, f1, f2)
        Save2Npy(file_dir=path1, save_dir=path2)


100%|██████████| 115/115 [05:49<00:00,  3.04s/it]


In [11]:
def getFPS(file_dir, save_dir):
    """Transfer all the videos and save them into specified directory
    Args:
        file_dir: source folder of target videos
        save_dir: destination folder of output .npy files
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    # List the files
    videos = os.listdir(file_dir)
    for v in videos:
        # Split video name
        video_name = v.split('.')[0]
        # Get src
        video_path = os.path.join(file_dir, v)

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps!=30:
            print(fps)

    return None

In [12]:
source_path = 'data/'
target_path = 'transformed_data'

for f1 in ['non_violent', 'violent']:
    for f2 in ['cam1']:
        path1 = os.path.join(source_path, f1, f2)
        path2 = os.path.join(target_path, f1, f2)
        getFPS(file_dir=path1, save_dir=path2)
